<a href="https://colab.research.google.com/github/ZanebRA/code-switching-codesaviours-si26-zaneb/blob/main/SI26_Week7_Zaneb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [ ]:
!pip install transformers torch datasets seqeval scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=50238f48cecdcd8d161f5eb0755deda1fa9e245c7485e97d7d5c8e8098f29f12
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving dataset.csv to dataset.csv


In [ ]:
import os

print(os.listdir())

['.config', 'dataset.csv', 'sample_data']


In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv('dataset.csv')

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (862, 3)

Columns:
['sentence', 'word', 'label']

First 5 rows:


,sentence,word,label
0,Aaj ka din bohot busy tha,Aaj,URD
1,Aaj ka din bohot busy tha,ka,URD
2,Aaj ka din bohot busy tha,din,URD
3,Aaj ka din bohot busy tha,bohot,URD
4,Aaj ka din bohot busy tha,busy,ENG


In [ ]:
print("Label distribution:")
print(df["label"].value_counts())

print("\nUnique labels:")
print(df["label"].unique())

Label distribution:
label
URD    459
ENG    403
Name: count, dtype: int64

Unique labels:
['URD' 'ENG']


In [ ]:
import pandas as pd

df = pd.read_csv("dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (862, 3)


,sentence,word,label
0,Aaj ka din bohot busy tha,Aaj,URD
1,Aaj ka din bohot busy tha,ka,URD
2,Aaj ka din bohot busy tha,din,URD
3,Aaj ka din bohot busy tha,bohot,URD
4,Aaj ka din bohot busy tha,busy,ENG


In [ ]:
# Group words and labels by sentence
sentences = df.groupby("sentence", sort=False).apply(
    lambda x: {
        "words": x["word"].tolist(),
        "labels": x["label"].tolist()
    }
).tolist()

print("Total sentences:", len(sentences))

print("\nFirst sentence:")
print(sentences[0])

Total sentences: 150

First sentence:
{'words': ['Aaj', 'ka', 'din', 'bohot', 'busy', 'tha'], 'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'URD']}


/tmp/ipykernel_1845/1598236254.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sentences = df.groupby("sentence", sort=False).apply(


In [ ]:
from sklearn.model_selection import train_test_split

# Split sentences into training and testing sets
train_data, test_data = train_test_split(
    sentences,
    test_size=0.2,
    random_state=42
)

print("Training sentences:", len(train_data))
print("Testing sentences:", len(test_data))

print("\nSample training data:")
print(train_data[0])

Training sentences: 120
Testing sentences: 30

Sample training data:
{'words': ['Tickets', 'bht', 'expensive', 'hain'], 'labels': ['ENG', 'URD', 'ENG', 'URD']}


In [ ]:
from transformers import AutoTokenizer
# Load XLM-RoBERTa tokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# Define label mapping
label2id = {
    "URD": 0,
    "ENG": 1
}

id2label = {
    0: "URD",
    1: "ENG"
}

print("Labels:", label2id)
print("Tokenizer loaded successfully!")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Labels: {'URD': 0, 'ENG': 1}
Tokenizer loaded successfully!


In [ ]:
# Tokenize one sample sentence
sample = train_data[0]

tokenized = tokenizer(
    sample["words"],
    is_split_into_words=True
)

print("Original words:")
print(sample["words"])

print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(tokenized["input_ids"]))

print("\nWord IDs:")
print(tokenized.word_ids())

Original words:
['Tickets', 'bht', 'expensive', 'hain']

Tokens:
['<s>', '▁Ticket', 's', '▁bh', 't', '▁expensive', '▁hain', '</s>']

Word IDs:
[None, 0, 0, 1, 1, 2, 3, None]


In [ ]:
def tokenize_and_align_labels(example):
    tokenized = tokenizer(
        example["words"],
        is_split_into_words=True,
        truncation=True,
        max_length=128
    )

    word_ids = tokenized.word_ids()
    labels = []

    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        else:
            labels.append(label2id[example["labels"][word_id]])

    tokenized["labels"] = labels

    return tokenized

In [ ]:
train_tokenized = [tokenize_and_align_labels(x) for x in train_data]
test_tokenized = [tokenize_and_align_labels(x) for x in test_data]

print("Training examples:", len(train_tokenized))
print("Testing examples:", len(test_tokenized))

print("\nFirst tokenized example:")
print(train_tokenized[0])

Training examples: 120
Testing examples: 30

First tokenized example:
{'input_ids': [0, 80304, 7, 14225, 18, 135587, 15531, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 1, 1, 0, 0, 1, 0, -100]}


In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

print("XLM-RoBERTa model loaded successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


XLM-RoBERTa model loaded successfully!


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_tokenized)
test_dataset = Dataset.from_list(test_tokenized)

print("Train dataset:")
print(train_dataset)

print("\nTest dataset:")
print(test_dataset)

Train dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 120
})

Test dataset:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 30
})


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./urdu-code-switching-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

print("Training arguments set successfully!")

Training arguments set successfully!


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

print("Trainer is ready!")

Trainer is ready!


In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

print("Data collator ready!")

Data collator ready!


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.625661,0.319438
2,0.244483,0.179093
3,0.162337,0.109091


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=45, training_loss=0.3349860986073812, metrics={'train_runtime': 423.6833, 'train_samples_per_second': 0.85, 'train_steps_per_second': 0.106, 'total_flos': 2339422525152.0, 'train_loss': 0.3349860986073812, 'epoch': 3.0})

In [ ]:
# Make predictions on the test dataset
predictions = trainer.predict(test_dataset)

print("Predictions generated successfully!")
print("Prediction shape:", predictions.predictions.shape)
print("Label shape:", predictions.label_ids.shape)

Predictions generated successfully!
Prediction shape: (30, 17, 2)
Label shape: (30, 17)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# Get predicted label IDs
pred_labels = np.argmax(predictions.predictions, axis=2)

# Get actual labels
true_labels = predictions.label_ids

# Remove special tokens and padding (-100)
pred_flat = []
true_flat = []

for pred_row, true_row in zip(pred_labels, true_labels):
    for pred, true in zip(pred_row, true_row):
        if true != -100:
            pred_flat.append(pred)
            true_flat.append(true)

# Calculate metrics
accuracy = accuracy_score(true_flat, pred_flat)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_flat,
    pred_flat,
    average="weighted"
)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

print("\nClassification Report:")
print(
    classification_report(
        true_flat,
        pred_flat,
        target_names=["URD", "ENG"]
    )
)

Accuracy: 0.9732
Precision: 0.9734
Recall: 0.9732
F1 Score: 0.9732

Classification Report:
              precision    recall  f1-score   support

         URD       0.98      0.97      0.98       125
         ENG       0.96      0.98      0.97        99

    accuracy                           0.97       224
   macro avg       0.97      0.97      0.97       224
weighted avg       0.97      0.97      0.97       224



In [ ]:
# Save the trained model and tokenizer
save_path = "./urdu-code-switching-model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!


In [ ]:
import torch

sentence = "Aaj mera mood bht good hai"

words = sentence.split()

inputs = tokenizer(
    words,
    is_split_into_words=True,
    return_tensors="pt",
    truncation=True
)

# Move inputs to the same device as the model
device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().tolist()

word_ids = tokenizer(
    words,
    is_split_into_words=True,
    truncation=True
).word_ids()

# Get one prediction for each original word
word_predictions = {}

for token_prediction, word_id in zip(predictions, word_ids):
    if word_id is not None and word_id not in word_predictions:
        word_predictions[word_id] = token_prediction

print("Predicted labels:")

for i, word in enumerate(words):
    label = id2label[word_predictions[i]]
    print(f"{word} → {label}")

Predicted labels:
Aaj → URD
mera → URD
mood → ENG
bht → URD
good → ENG
hai → URD


In [ ]:
# Login to Hugging Face Hub
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
user_info = api.whoami()

print("Hugging Face username:", user_info["name"])

Hugging Face username: zaneb-217


In [ ]:
from huggingface_hub import HfApi

repo_id = "zaneb-217/urdu-code-switching-model"

api = HfApi()

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True
)

print("Model repository created:", repo_id)

Model repository created: zaneb-217/urdu-code-switching-model


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="./urdu-code-switching-model",
    repo_id="zaneb-217/urdu-code-switching-model",
    repo_type="model"
)

print("Model uploaded successfully!")

Model uploaded successfully!


In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("zaneb-217/urdu-code-switching-model")

print("Files in Hugging Face repository:")
for file in files:
    print(file)

Files in Hugging Face repository:
.gitattributes
checkpoint-15/config.json
checkpoint-15/model.safetensors
checkpoint-15/optimizer.pt
checkpoint-15/rng_state.pth
checkpoint-15/scheduler.pt
checkpoint-15/tokenizer.json
checkpoint-15/tokenizer_config.json
checkpoint-15/trainer_state.json
checkpoint-15/training_args.bin
checkpoint-30/config.json
checkpoint-30/model.safetensors
checkpoint-30/optimizer.pt
checkpoint-30/rng_state.pth
checkpoint-30/scheduler.pt
checkpoint-30/tokenizer.json
checkpoint-30/tokenizer_config.json
checkpoint-30/trainer_state.json
checkpoint-30/training_args.bin
checkpoint-45/config.json
checkpoint-45/model.safetensors
checkpoint-45/optimizer.pt
checkpoint-45/rng_state.pth
checkpoint-45/scheduler.pt
checkpoint-45/tokenizer.json
checkpoint-45/tokenizer_config.json
checkpoint-45/trainer_state.json
checkpoint-45/training_args.bin
config.json
model.safetensors
tokenizer.json
tokenizer_config.json
training_args.bin


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

hf_model = "zaneb-217/urdu-code-switching-model"

hf_tokenizer = AutoTokenizer.from_pretrained(hf_model)
hf_model_loaded = AutoModelForTokenClassification.from_pretrained(hf_model)

print("Model loaded successfully from Hugging Face!")
print("Labels:", hf_model_loaded.config.id2label)

config.json:   0%|          | 0.00/860 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded successfully from Hugging Face!
Labels: {0: 'URD', 1: 'ENG'}


In [ ]:
import torch

sentence = "Aaj meri meeting bohot important hai"

words = sentence.split()

inputs = hf_tokenizer(
    words,
    is_split_into_words=True,
    return_tensors="pt",
    truncation=True
)

with torch.no_grad():
    outputs = hf_model_loaded(**inputs)

predictions = torch.argmax(outputs.logits, dim=-1)[0]

word_ids = inputs.word_ids()

print("Predicted labels:")

previous_word_id = None

for token_index, word_id in enumerate(word_ids):
    if word_id is not None and word_id != previous_word_id:
        label_id = predictions[token_index].item()
        label = hf_model_loaded.config.id2label[label_id]
        print(f"{words[word_id]} → {label}")
        previous_word_id = word_id

Predicted labels:
Aaj → URD
meri → URD
meeting → ENG
bohot → URD
important → ENG
hai → URD


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

old_model = "zaneb-217/urdu-code-switching-model"

tokenizer = AutoTokenizer.from_pretrained(old_model)
model = AutoModelForTokenClassification.from_pretrained(old_model)

print("Trained model loaded successfully!")
print("Labels:", model.config.id2label)

config.json:   0%|          | 0.00/860 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Trained model loaded successfully!
Labels: {0: 'URD', 1: 'ENG'}


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

repo_name = "code-switching-codesaviours-si26-zaneb"

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print(f"Model published at: https://huggingface.co/zaneb-217/{repo_name}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...jgwznsy/model.safetensors:   3%|2         | 32.0MB / 1.11GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpisfb830o/tokenizer.json:  47%|####6     | 7.96MB / 17.1MB            

Model published at: https://huggingface.co/zaneb-217/code-switching-codesaviours-si26-zaneb
